# 20. 두 시기 사진을 섞어 학습하면 가짜 변화가 줄어드나 (실험 8)

**한 줄 요약:** 17–19번에서 숲이 그대로인데 2022–23과 2024–25 사진의 탄소가 지역 평균 1.28%p 달랐고(가짜 변화), 원인은 여름 사진이었으며 구름 마스크·새 위성·달 구성으로는 줄지 않았다. 사진을 고치는 대신 **모델에게 두 해 사진을 다 보여 주고 "둘 다 같은 숲"이라고 가르치면** 가짜 변화가 줄어드는지 본다. 새 데이터 없음.

**설계 (05·17과 한 가지만 다름)**
- 학습 픽셀은 05와 **같은 픽셀, 같은 개수**(지역당 3만, 같은 난수). 다른 점은 픽셀마다 **동전을 던져 절반은 2022–23 사진 값**으로 넣는 것뿐
- 평가는 17과 같음: 처음 보는 지역, 숲이 그대로인 픽셀의 두 시기 탄소 차이
- 대가 확인: 2024–25 사진의 탄소 오차(05·09 값)가 나빠지지 않는지

**주의:** 시험하는 두 해를 다른 지역 학습에서 이미 봤으므로 **후한 점수**다. 통과하면 처음 보는 해(2020–21)로 다시 시험해야 한다(사전 기록 다음 단계).

**"모두 실행" 가능.** [2]에서 `CONFIRMED = True`로 바꾸기 전에는 멈춘다. 폴드마다 저장 → 끊기면 다시 "모두 실행". 약 1.5–2.5시간 [추정], 첫 폴드 후 남은 시간 출력.

In [2]:
# [1] 설정
!pip install -q rasterio

import os, json, time, gc, warnings
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from google.colab import drive
warnings.filterwarnings('ignore', category=RuntimeWarning)

drive.mount('/content/drive')
OUT      = '/content/drive/MyDrive/forest/outputs'
FOLD_DIR = f'{OUT}/exp8_folds'
PR_PATH  = f'{OUT}/exp8_prereg.json'
os.makedirs(FOLD_DIR, exist_ok=True)

TRAIN_REGIONS = ['inje', 'hongcheon', 'gangneung', 'pyeongchang',
                 'bonghwa', 'geumsan', 'geochang', 'suncheon']
KO = {'inje': '인제', 'hongcheon': '홍천', 'gangneung': '강릉', 'pyeongchang': '평창',
      'bonghwa': '봉화', 'geumsan': '금산', 'geochang': '거창', 'suncheon': '순천'}
C = {1: 79.25, 2: 101.02, 3: 92.01}            # 확정계수만
SEEDS              = [16, 17, 18, 19, 20]
N_TRAIN_PER_REGION = 30_000
MIX_P              = 0.5                        # 학습 픽셀 중 2022–23 사진으로 넣는 비율
RF_PARAMS = dict(n_estimators=80, min_samples_leaf=20, max_features='sqrt',
                 class_weight='balanced', n_jobs=-1)     # 05와 동일

for r in TRAIN_REGIONS:
    for f in (f'{OUT}/{r}_s05.npz', f'{OUT}/{r}_old.npz'):
        assert os.path.exists(f), f'없음: {f}'
    for s in SEEDS:
        assert os.path.exists(f'{OUT}/exp4c_folds/s{s}_{r}.npz'), f'09 폴드 없음: s{s}_{r}'
assert os.path.exists(f'{OUT}/exp7_results.npz')
print('파일 확인 완료')

Mounted at /content/drive
파일 확인 완료


In [3]:
# [2] 사전 기록 — 결과를 보기 전에 확정. CONFIRMED = True 로 바꿔야 진행된다
CONFIRMED = True

PREREG = {
    'date': time.strftime('%Y-%m-%d %H:%M'),
    'state_before': ('17(가짜 변화 지역 |Δ| 평균 1.28%p, 모두 +), 18(여름 주원인, 겨울은 반대로 일부 상쇄, 눈 아님), '
                     '19(Cloud Score+·S2C 제외·달 균등 합성 모두 지지 안 됨, 판정 "없음") 결과를 봄. 이 노트북 결과는 보지 않음'),
    'question': '학습 때 두 시기 사진을 섞어 보여 주면, 처음 보는 지역의 가짜 변화가 줄어드는가. 그 대가로 2024–25 정확도가 나빠지는가',
    'design': ('05와 같은 LORO 40폴드·같은 RF·같은 학습 픽셀(05 난수 그대로). 픽셀마다 별도 난수(seed+1000)로 '
               f'{MIX_P:.0%} 확률로 2022–23 값 사용(2022–23 값 있음 & Hansen lossyear < 22인 픽셀만, 아니면 2024–25). '
               '평가 = 17과 같은 픽셀(ok & lossyear < 22), 17과 같은 오차 정의. '
               '정확도 = 평가 지역 전체 2024–25 표본의 탄소 오차(05/09 carb0와 같은 정의)와 macro F1'),
    'rule_improve': ('개선 = 지역 |Δ| 평균(8곳)이 17의 같은 시드 값의 절반 이하 — 5/5 시드. '
                     '근거: 18·19와 같은 "반 이상 사라짐" 기준'),
    'rule_usable': ('실험 7 기준 통과 = 지역 |Δ| 평균 < 0.5%p AND 8곳 모두 |Δ| < 1.0%p — 5/5 시드 (17과 같은 규칙)'),
    'rule_cost': ('정확도 대가 있음 = 2024–25 지역 |오차| 평균이 09(같은 시드)보다 0.2%p 넘게 나빠진 시드가 4/5 이상. '
                  '근거: 13a 계절 사진 추가로 얻은 개선이 약 0.16%p — 그만큼 잃으면 실험 하나만큼 후퇴'),
    'record_only': ['지역별 Δ, 침엽 예측 비율(두 시기)', 'macro F1 (09 확률로 만든 05 F1과 비교)',
                    '면적가중 합산 Δ', '학습 픽셀 중 실제로 2022–23 값이 들어간 비율'],
    'caveat': '두 시험 시기를 다른 지역 학습에서 이미 봄 → 후한 점수. 처음 보는 해에서의 성능은 이 실험으로 말하지 않음',
    'predictions_추정': ['가짜 변화는 줄지만 절반 이하까지는 못 갈 수 있음 (지역 |Δ| 평균 0.5–0.9)',
                         '정확도 대가는 작음 (0.2%p 이내)'],
    'next_step': {
        '개선 + 대가 없음': ('21 = 처음 보는 해 시험: 2020–21 사진을 16번 방식으로 받기 → 두 시기(2022–23·2024–25)로 학습, '
                          '2024–25 vs 2020–21 가짜 변화를 잼 (진짜 시험)'),
        '개선 + 대가 있음': '섞는 비율(MIX_P)이나 표본 수를 바꾸기 전에 사용자와 상의 — 정확도와 연도 안정성 맞바꿈',
        '개선 아님': ('모델 쪽으로도 영점 차이를 못 지움 → 변화량 계산 방식 전환: 지도 두 장 빼기 대신 '
                     '"변한 곳만 찾아 갱신"(IPCC 늘어남−줄어듦 방식). 별도 사전 기록'),
    },
    'seeds': SEEDS, 'mix_p': MIX_P, 'carbon_coeff': C,
    'rf': {k: v for k, v in RF_PARAMS.items() if k != 'n_jobs'},
    'checks': ['첫 폴드에서 05 순수 모델 재현 = 09 carb0 (0.01%p)', '평가 픽셀 수 = 17 n_eval'],
}

if os.path.exists(PR_PATH):
    print('저장된 사전 기록 (덮어쓰지 않음):\n')
    print(json.dumps(json.load(open(PR_PATH, encoding='utf-8')), ensure_ascii=False, indent=1))
else:
    assert CONFIRMED, '사전 기록을 읽고 CONFIRMED = True 로 바꾼 뒤 다시 실행하세요 (정상 정지)'
    json.dump(PREREG, open(PR_PATH, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
    print('사전 기록 저장:', PR_PATH)

사전 기록 저장: /content/drive/MyDrive/forest/outputs/exp8_prereg.json


In [4]:
# [3] 캐시 읽기 + 함수
def carbon_sum(y):
    return float(sum((y == v).sum() * C[v] for v in (1, 2, 3)))

def carbon_err(y_true, y_pred):
    t = carbon_sum(y_true)
    return (carbon_sum(y_pred) - t) * 100 / t

S, O, MASK = {}, {}, {}
E7 = np.load(f'{OUT}/exp7_results.npz')
assert [str(x) for x in E7['regions']] == TRAIN_REGIONS and [int(x) for x in E7['seeds']] == SEEDS
for j, r in enumerate(TRAIN_REGIONS):
    z = np.load(f'{OUT}/{r}_s05.npz'); o = np.load(f'{OUT}/{r}_old.npz')
    assert (o['y'] == z['y']).all(), f'{r}: s05와 old 픽셀 순서 다름'
    S[r] = dict(X=z['X'], y=z['y'])
    O[r] = o['X']
    MASK[r] = o['ok'] & (o['loss'] < 22)
    n7 = int(round(float(E7['n_eval'][0, j])))
    assert int(MASK[r].sum()) == n7, f'{r}: 평가 픽셀 {int(MASK[r].sum()):,} ≠ 17의 {n7:,}'
print('캐시 읽기·검산 완료 (17과 같은 평가 픽셀)')

def train_set(seed, test, mix):
    rng = np.random.default_rng(seed)                     # 05와 같은 픽셀 뽑기
    rng_mix = np.random.default_rng(seed + 1000)          # 사진 해 고르기 (별도 난수 → 픽셀 집합은 05 그대로)
    Xtr, ytr, n_old, n_all = [], [], 0, 0
    for r in TRAIN_REGIONS:
        if r == test: continue
        n = len(S[r]['y'])
        i = rng.choice(n, size=min(N_TRAIN_PER_REGION, n), replace=False)
        X = S[r]['X'][i].copy()
        if mix:
            use_old = MASK[r][i] & (rng_mix.random(len(i)) < MIX_P)
            X[use_old] = O[r][i[use_old]]
            n_old += int(use_old.sum())
        n_all += len(i)
        Xtr.append(X); ytr.append(S[r]['y'][i])
    return np.concatenate(Xtr), np.concatenate(ytr), n_old / n_all

# 검산: 첫 폴드에서 순수 05 모델 재현 (한 번만, 1–3분)
s0, r0 = SEEDS[0], TRAIN_REGIONS[0]
if not os.path.exists(f'{FOLD_DIR}/check_05.json'):
    Xtr, ytr, _ = train_set(s0, r0, mix=False)
    clf = RandomForestClassifier(random_state=s0, **RF_PARAMS).fit(Xtr, ytr)
    e = carbon_err(S[r0]['y'], clf.predict(S[r0]['X']))
    ref = float(np.load(f'{OUT}/exp4c_folds/s{s0}_{r0}.npz')['carb0'])
    assert abs(e - ref) < 0.01, f'05 재현 실패 {e:.3f} vs 09 {ref:.3f} — 멈추고 보고'
    json.dump(dict(err=e, ref=ref), open(f'{FOLD_DIR}/check_05.json', 'w'))
    del clf, Xtr, ytr; gc.collect()
print('05 재현 검산 통과:', json.load(open(f'{FOLD_DIR}/check_05.json')))

캐시 읽기·검산 완료 (17과 같은 평가 픽셀)
05 재현 검산 통과: {'err': 2.429857583296202, 'ref': 2.429857583296202}


In [ ]:
# [4] 실험 8 — LORO 40폴드 (폴드마다 저장, 끊기면 다시 실행)
assert os.path.exists(PR_PATH), '사전 기록 [2]를 먼저 확정하세요 (정상 정지)'
folds = [(s, r) for s in SEEDS for r in TRAIN_REGIONS]
todo  = [(s, r) for s, r in folds if not os.path.exists(f'{FOLD_DIR}/s{s}_{r}.npz')]
print(f'전체 {len(folds)}폴드 중 완료 {len(folds) - len(todo)}, 남은 {len(todo)}\n')

times = []
for k, (seed, test) in enumerate(todo, 1):
    t0 = time.time()
    i_s, j_r = SEEDS.index(seed), TRAIN_REGIONS.index(test)
    Xtr, ytr, old_share = train_set(seed, test, mix=True)
    clf = RandomForestClassifier(random_state=seed, **RF_PARAMS).fit(Xtr, ytr)
    del Xtr, ytr; gc.collect()

    y_all = S[test]['y']
    p_all = clf.predict(S[test]['X'])
    e_full = carbon_err(y_all, p_all)
    f1 = f1_score(y_all, p_all, average='macro', labels=[1, 2, 3])
    del p_all

    m = MASK[test]; y = y_all[m]
    p_new = clf.predict(S[test]['X'][m]); p_old = clf.predict(O[test][m])
    e_new, e_old = carbon_err(y, p_new), carbon_err(y, p_old)
    c_new, c_old = (p_new == 1).mean() * 100, (p_old == 1).mean() * 100

    np.savez(f'{FOLD_DIR}/s{seed}_{test}.npz', e_full=e_full, f1=f1, e_new=e_new, e_old=e_old,
             c_new=c_new, c_old=c_old, old_share=old_share, n_eval=int(m.sum()))
    del clf, p_new, p_old; gc.collect()

    dt = time.time() - t0; times.append(dt)
    print(f'[{k:2d}/{len(todo)}] s{seed} {KO[test]}  가짜변화 {e_new - e_old:+.2f} (17: {E7["dh"][i_s, j_r]:+.2f})  '
          f'2024–25 오차 {e_full:+.2f}  F1 {f1:.3f}  2022–23 섞인 비율 {old_share:.0%}   '
          f'{dt:.0f}초  남은 약 {np.mean(times) * (len(todo) - k) / 60:.0f}분')
print('\n모든 폴드 완료')

전체 40폴드 중 완료 0, 남은 40

[ 1/40] s16 인제  가짜변화 +2.16 (17: +3.01)  2024–25 오차 +2.65  F1 0.599  2022–23 섞인 비율 49%   169초  남은 약 110분
[ 2/40] s16 홍천  가짜변화 +1.30 (17: +1.95)  2024–25 오차 +1.57  F1 0.616  2022–23 섞인 비율 49%   158초  남은 약 104분
[ 3/40] s16 강릉  가짜변화 -0.02 (17: +0.16)  2024–25 오차 -0.32  F1 0.609  2022–23 섞인 비율 49%   163초  남은 약 101분
[ 4/40] s16 평창  가짜변화 -0.07 (17: -0.07)  2024–25 오차 +3.14  F1 0.599  2022–23 섞인 비율 49%   157초  남은 약 97분
[ 5/40] s16 봉화  가짜변화 -0.04 (17: +0.53)  2024–25 오차 +1.32  F1 0.592  2022–23 섞인 비율 49%   159초  남은 약 94분
[ 6/40] s16 금산  가짜변화 +0.79 (17: +1.50)  2024–25 오차 +2.56  F1 0.507  2022–23 섞인 비율 49%   156초  남은 약 91분
[ 7/40] s16 거창  가짜변화 +1.15 (17: +1.37)  2024–25 오차 +0.46  F1 0.632  2022–23 섞인 비율 49%   157초  남은 약 88분
[ 8/40] s16 순천  가짜변화 +0.94 (17: +1.54)  2024–25 오차 -1.40  F1 0.556  2022–23 섞인 비율 49%   156초  남은 약 85분
[ 9/40] s17 인제  가짜변화 +2.14 (17: +2.92)  2024–25 오차 +2.60  F1 0.600  2022–23 섞인 비율 49%   159초  남은 약 82분
[10/40] s17 홍천  가짜변화 +1.31 (17: +1.97)  2024–25

In [ ]:
# [5] 모으기 + 사전 기록 규칙대로 판정
ns, nr = len(SEEDS), len(TRAIN_REGIONS)
keys = ['e_full', 'f1', 'e_new', 'e_old', 'c_new', 'c_old', 'old_share']
R = {k: np.full((ns, nr), np.nan) for k in keys}
CARB0, F1_05 = np.full((ns, nr), np.nan), np.full((ns, nr), np.nan)
for i, s in enumerate(SEEDS):
    for j, r in enumerate(TRAIN_REGIONS):
        f = f'{FOLD_DIR}/s{s}_{r}.npz'
        if os.path.exists(f):
            z = np.load(f)
            for k in keys: R[k][i, j] = z[k]
        z9 = np.load(f'{OUT}/exp4c_folds/s{s}_{r}.npz')
        CARB0[i, j] = float(z9['carb0'])
        if 'proba' in z9.files and len(z9['proba']) == len(S[r]['y']):
            F1_05[i, j] = f1_score(S[r]['y'], np.array([1, 2, 3])[z9['proba'].argmax(1)],
                                   average='macro', labels=[1, 2, 3])
assert not np.isnan(R['e_new']).any(), '미완료 폴드 — [4]를 다시 실행'

DM, D7 = R['e_new'] - R['e_old'], E7['dh']
tab = pd.DataFrame({'가짜변화_17': D7.mean(0), '가짜변화_섞음': DM.mean(0),
                    '침엽%_신': R['c_new'].mean(0), '침엽%_구': R['c_old'].mean(0),
                    '오차2024–25_09': CARB0.mean(0), '오차2024–25_섞음': R['e_full'].mean(0),
                    'F1_05': F1_05.mean(0), 'F1_섞음': R['f1'].mean(0)},
                   index=[KO[r] for r in TRAIN_REGIONS])
tab.to_csv(f'{OUT}/exp8_by_region.csv')
pd.set_option('display.width', 250)
print('5시드 평균 (%p / %)\n'); print(tab.round(3).to_string())

m17, mM = np.abs(D7).mean(1), np.abs(DM).mean(1)
a09, aM = np.abs(CARB0).mean(1), np.abs(R['e_full']).mean(1)
print('\n시드별  지역 |가짜변화| 평균  17 → 섞음 :  ' + '  '.join(f'{a:.2f}→{b:.2f}' for a, b in zip(m17, mM)))
print('시드별  2024–25 지역 |오차| 평균 09 → 섞음 :  ' + '  '.join(f'{a:.2f}→{b:.2f}' for a, b in zip(a09, aM)))
print(f'[기록] 학습 픽셀 중 2022–23 값 비율 {np.nanmean(R["old_share"]):.1%}')

improve = bool((mM <= 0.5 * m17).all())
usable  = bool(((mM < 0.5) & (np.abs(DM).max(1) < 1.0)).all())
cost    = int(((aM - a09) > 0.2).sum()) >= 4
print('\n' + '=' * 60)
print(f'개선(절반 이하, 5/5):        {"예" if improve else "아니오"}  ({int((mM <= 0.5 * m17).sum())}/5)')
print(f'실험 7 기준 통과(5/5):       {"예" if usable else "아니오"}')
print(f'정확도 대가(0.2%p 넘게, ≥4/5): {"있음" if cost else "없음"}  ({int(((aM - a09) > 0.2).sum())}/5)')
pr = json.load(open(PR_PATH, encoding='utf-8'))
key = ('개선 + 대가 있음' if cost else '개선 + 대가 없음') if improve else '개선 아님'
print('사전 기록의 다음 단계:', pr['next_step'][key])
print('=' * 60)
json.dump(dict(improve=improve, usable=usable, cost=cost, abs_dh_17=m17.tolist(), abs_dh_mix=mM.tolist(),
               abs_err_09=a09.tolist(), abs_err_mix=aM.tolist()),
          open(f'{OUT}/exp8_judgement.json', 'w'), indent=1)